# Fasl · Notebook 1: Collect and clean wheat data
Merges every attached dataset into one clean, de-duplicated, split dataset for two tasks:
- **health**: healthy, brown_rust, yellow_rust, stem_rust, septoria, loose_smut, powdery_mildew
- **stage**: vegetative, flowering, grain_filling, ripening

plus a **not_wheat** reject class.

**How to run:** run Steps 1 to 10 in order, once. Output goes to `/kaggle/working/fasl_clean/`.
Then **Save Version → Save & Run All**, and turn the output into a dataset named `fasl-clean` for Notebook 2.

## Step 1: Settings and attached datasets
Imports, all settings (image size, blur limit, duplicate limits, split ratio) and the list of attached datasets.
Run this first every time the session starts.

In [ ]:
import os, re, json, hashlib, random, shutil
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.fft import dctn

INPUT_ROOT = Path(os.getenv("FASL_INPUT", "/kaggle/input"))
OUT = Path(os.getenv("FASL_OUT", "/kaggle/working/fasl_clean"))
TMP = Path(os.getenv("FASL_TMP", "/kaggle/tmp/fasl_resized"))  # resized copies on disk, not in RAM
SEED = 42
MAX_SIDE = 512            # resize longest side to this
MIN_SIDE = 64             # drop images whose short side is below this
BLUR_MIN = 15.0           # Laplacian variance below this = too blurry
DUP_DIST = 4              # pHash distance <= this = duplicate (keep one)
GROUP_DIST = 7            # pHash distance <= this = same scene (kept in same split)
MIN_PER_CLASS = 15        # classes below this are dropped
MAX_PER_SOURCE_CLASS = 1500
MAX_NOT_WHEAT = 4500     # spread over the three not_wheat sources
SPLIT = (0.70, 0.15, 0.15)
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
random.seed(SEED); np.random.seed(SEED)

# Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<name>/ (older layout: /kaggle/input/<name>/)
DS = {}
for d in sorted(INPUT_ROOT.glob("datasets/*/*")) + sorted(INPUT_ROOT.glob("competitions/*")):
    if d.is_dir(): DS[d.name] = d
for d in sorted(INPUT_ROOT.iterdir()):
    if d.is_dir() and d.name not in {"datasets", "notebooks", "competitions", "models"}: DS.setdefault(d.name, d)
print("Attached datasets:")
for k, v in DS.items(): print(f"   {k:45s} {v}")
if not DS:
    print("   NONE. Attach the datasets: right-hand panel > Add Input (see Step 2 for the list).")
    print("What is under", INPUT_ROOT, ":")
    for p in sorted(INPUT_ROOT.glob("*"))[:20] + sorted(INPUT_ROOT.glob("*/*"))[:40] + sorted(INPUT_ROOT.glob("*/*/*"))[:40]:
        print("   ", p)

## Step 2: Which dataset is what
`health` = disease images, `stage` = growth-stage images, `not_wheat` = reject class. Datasets not listed are ignored.
`ALSO_HEALTHY` = stage datasets of disease-free plants; their photos are also used as `health/healthy`, so the health model learns what a healthy ear and field look like.

**Healthy ears (recommended, optional):** attach the Kaggle competition `global-wheat-detection`.
Its field photos of wheat heads are added as `health/healthy`. Without enough healthy ear photos the model
calls every ear `loose_smut`, because loose smut is the only disease class photographed on ears.

**not_wheat sources** (attach all three in Kaggle: Add Input):
- `plantdisease` (emmarex): other crops' leaves
- `natural-images` (prasunroy): people, cars, animals, flowers, fruit
- `intel-image-classification` (puneet6060): buildings, streets, forest, mountains, sea

Without the last two the model has never seen a person or a room, and calls them wheat.

**More growth-stage data** (strongly recommended, the stage model has very few photos without them):
- `gwhd-stages`: upload `gwhd-stages.zip` (made by `ml/scripts/make_gwhd_stages.py` from the Global Wheat Head Dataset 2021,
  CC-BY 4.0) as a Kaggle dataset with that name. Field photos at flowering, grain filling and ripening; also used as healthy.
- `wheat-weed-field-image-dataset` (sarfarazkhanmphil): young wheat in Peshawar, 15-36 days, as `vegetative`.
  Only folders named `Wheat` are used; weed folders are skipped (listed under UNMAPPED).

In [ ]:
# Step 2: Which dataset is what
# Finds the datasets itself (does not depend on Step 1), so re-running only this cell is fine.
import re
from pathlib import Path

INPUT_ROOT = Path(os.getenv("FASL_INPUT", "/kaggle/input"))

SOURCES = {
    "wheatphenology": "stage",
    "gwhd-stages": "stage",                      # Global Wheat Head Dataset 2021 by stage (fasl upload)
    "wheat-weed-field-image-dataset": "stage",   # young wheat 15-36 days, Peshawar (wheat photos only)
    "wheat-leaf-disease-dataset": "health",
    "wheat-plant-diseases": "health",
    "wheat-leaf-dataset": "health",
    "yellowrust19-yellow-rust-disease-in-wheat": "health",
    "plantdisease": "not_wheat",                 # other crops' leaves
    "natural-images": "not_wheat",               # people, cars, animals, flowers, fruit
    "intel-image-classification": "not_wheat",   # buildings, streets, forest, mountains, sea
}
# Stage datasets of disease-free plants: their photos are ALSO used as health/healthy, so the health
# model learns healthy ears and fields. Without this, every ear photo looks like loose_smut.
ALSO_HEALTHY = {"wheatphenology", "gwhd-stages", "wheat-weed-field-image-dataset"}
# Optional extra healthy ears (Competitions tab: "global-wheat-detection"). Skipped if not attached.
OPTIONAL_HEALTHY = ["global-wheat-detection", "global-wheat-head-dataset-2021", "gwhd-2021"]

def _key(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

# Every folder up to 3 levels deep: /kaggle/input/<name>, .../datasets/<owner>/<name>, .../competitions/<name>
found = {}
level = [INPUT_ROOT] if INPUT_ROOT.exists() else []
for _ in range(3):
    nxt = []
    for d in level:
        try:
            subs = [p for p in d.iterdir() if p.is_dir()]
        except OSError:
            continue
        for p in subs:
            found.setdefault(_key(p.name), p)   # shallowest match wins
        nxt += subs
    level = nxt

DS = {name: found[_key(name)] for name in list(SOURCES) + OPTIONAL_HEALTHY if _key(name) in found}

print("Datasets:")
for name in list(SOURCES) + OPTIONAL_HEALTHY:
    print(f"   {'OK     ' if name in DS else 'MISSING'}  {name:45s} {DS.get(name, '')}")

health_found = [s for s, t in SOURCES.items() if t == "health" and s in DS]
not_wheat_found = [s for s, t in SOURCES.items() if t == "not_wheat" and s in DS]
problems = []
if "wheatphenology" not in DS: problems.append("wheatphenology (needed for growth stage and healthy ears)")
if not health_found: problems.append("at least one wheat disease dataset")
if not not_wheat_found: problems.append("at least one not_wheat dataset, e.g. plantdisease")
if problems:
    print("\nFolders found under", INPUT_ROOT, ":")
    for p in sorted({str(v) for v in found.values()})[:60]: print("   ", p)
    raise SystemExit("STOP. Attach in the right-hand panel > Add Input: " + "; ".join(problems))

SOURCES = {k: v for k, v in SOURCES.items() if k in DS}
HEALTHY_EAR_SOURCES = ALSO_HEALTHY | {s for s in OPTIONAL_HEALTHY if s in DS}
print("\nUsing:", list(SOURCES))
print("Healthy-ear sources:", sorted(HEALTHY_EAR_SOURCES))
if not any(s in DS for s in OPTIONAL_HEALTHY):
    print("Tip: attach the competition 'global-wheat-detection' for more healthy ear photos.")

## Step 3: Label mapping and image list
Each image's label comes from its folder name, mapped to our classes below.
- Yellow Rust 19 severity: `0` = healthy; `MS`, `S`, `MRMS` = yellow_rust; `R`, `MR` skipped (too few symptoms).
- Diseases we don't predict (aphid, mite, blast, tan spot, ...) are skipped and listed under UNMAPPED.
- Max 1,500 images per class per source, max 3,000 not_wheat.

In [ ]:
HEALTH = ["healthy","brown_rust","yellow_rust","stem_rust","septoria","loose_smut","powdery_mildew"]
STAGE = ["vegetative","flowering","grain_filling","ripening"]

ALIASES = {
  "health": {
    "healthy":"healthy","healthy_wheat":"healthy","healthy_leaf":"healthy","normal":"healthy","0":"healthy",
    "brown_rust":"brown_rust","leaf_rust":"brown_rust","brownrust":"brown_rust",
    "yellow_rust":"yellow_rust","stripe_rust":"yellow_rust","yellowrust":"yellow_rust",
    "ms":"yellow_rust","s":"yellow_rust","mrms":"yellow_rust",
    "stem_rust":"stem_rust","black_rust":"stem_rust","stemrust":"stem_rust",
    "septoria":"septoria","septoria_leaf_blotch":"septoria",
    "loose_smut":"loose_smut","smut":"loose_smut",
    "powdery_mildew":"powdery_mildew","mildew":"powdery_mildew",
  },
  "stage": {
    "seedling_and_plant":"vegetative", "seedling_plant":"vegetative",   # "1. Seedling & Plant"
    "wheat_flowers":"flowering",
    "plant_with_fruit":"grain_filling",   # TEMP: confirm after viewing images
    "fruit_with_seeds":"ripening",        # TEMP: confirm after viewing images
    # gwhd-stages (GWHD 2021 development stage per sub-dataset)
    "postflowering":"flowering", "filling":"grain_filling", "ripening":"ripening",
    # wheat-weed-field-image-dataset: wheat seedlings 15-36 days; weed folders stay unmapped
    "wheat":"vegetative", "wheats":"vegetative",   # folder is "Wheats"
  },
  "not_wheat": None,                      # every image becomes "not_wheat"
}
GENERIC = {"train","test","val","valid","validation","images","image","img","data","dataset","imgs","jpg"}
SKIP = {"r","mr"}

def norm(s):
    s = re.sub(r"^\d+\.?[\s_-]+", "", s.strip().lower())         # "1. Seedling" -> "seedling"
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    s = re.sub(r"(_test)?(_valid|_test|_train)$", "", s)          # "smut_test" -> "smut"
    return s or "0"

def label_for(path, task):
    if ALIASES[task] is None: return "not_wheat"
    for part in reversed(path.parent.parts):
        n = norm(part)
        if n in GENERIC: continue
        if n in SKIP: return None
        return ALIASES[task].get(n)
    return None

rows, unmapped = [], Counter()
for src, task in SOURCES.items():
    for p in DS[src].rglob("*"):
        if p.suffix.lower() not in IMG_EXT: continue
        lab = label_for(p, task)
        if lab is None: unmapped[(src, p.parent.name)] += 1; continue
        rows.append(dict(src_path=str(p), source=src, task=task, label=lab))
rows += [dict(r, task="health", label="healthy") for r in list(rows)
         if r["source"] in ALSO_HEALTHY and r["task"] == "stage"]
for src in HEALTHY_EAR_SOURCES - ALSO_HEALTHY:
    rows += [dict(src_path=str(p), source=src, task="health", label="healthy")
             for p in DS[src].rglob("*") if p.suffix.lower() in IMG_EXT]
raw = pd.DataFrame(rows).sample(frac=1, random_state=SEED)
raw = raw.groupby(["source","label"]).head(MAX_PER_SOURCE_CLASS)
raw = pd.concat([raw[raw.task != "not_wheat"], raw[raw.task == "not_wheat"].head(MAX_NOT_WHEAT)], ignore_index=True)

print(raw.groupby(["task","label"]).size().to_string())
print("\nUNMAPPED (skipped) folders:")
for k, v in unmapped.most_common(): print("  ", k, v)

## Step 4: Check, fix and resize every image (slow, run once)
Opens each image, fixes rotation, converts to RGB, drops broken or tiny images, resizes to 512 px and saves a copy to disk.
Also computes a hash (for duplicates) and a blur score. Takes 5 to 15 minutes.

In [ ]:
def phash(img):
    g = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float32)
    d = dctn(g, norm="ortho")[:8, :8]
    bits = (d > np.median(d)).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def blur_score(img):
    g = np.asarray(img.convert("L").resize((256, 256)), dtype=np.float32)
    lap = (-4*g[1:-1,1:-1] + g[:-2,1:-1] + g[2:,1:-1] + g[1:-1,:-2] + g[1:-1,2:])
    return float(lap.var())

def process(src_path):
    try:
        b = Path(src_path).read_bytes()
        img = Image.open(src_path); img.load()
        img = ImageOps.exif_transpose(img).convert("RGB")
    except Exception:
        return {"status": "broken"}
    if min(img.size) < MIN_SIDE:
        return {"status": "too_small"}
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    sha = hashlib.sha256(b).hexdigest()
    tmp = TMP / f"{sha}.jpg"
    if not tmp.exists(): img.save(tmp, quality=92)
    return {"status": "ok", "sha": sha, "phash": phash(img), "blur": blur_score(img),
            "w": img.width, "h": img.height, "tmp": str(tmp)}

TMP.mkdir(parents=True, exist_ok=True)
log = Counter(); recs = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for i, (r, res) in enumerate(zip(raw.itertuples(index=False), ex.map(process, raw.src_path, chunksize=64))):
        if res["status"] != "ok": log[res["status"]] += 1; continue
        res.pop("status"); recs.append({**r._asdict(), **res})
        if i % 2000 == 0: print(f"  {i}/{len(raw)} done")
df = pd.DataFrame(recs)
print(dict(log), "| valid:", len(df))
print(df.groupby(["task","label"]).size().to_string())

## Step 5: Remove duplicate images
- Same or nearly the same image in one class: keep one copy.
- Same image in two different classes: label conflict, remove all copies.
- Groups similar-looking photos so they stay together in the train / val / test split.

In [ ]:
DUP_DIST = 2   # stricter than before: only near-identical copies count as duplicates

def bucket_keys(h, maxd):
    k = maxd + 1; size = 64 // k
    return [(c, (h >> (c*size)) & ((1 << size) - 1)) for c in range(k)]

before = len(df); kept = []
for task, d in df.groupby("task"):
    d = d.sort_values("blur", ascending=False).reset_index(drop=True)   # sharpest copy wins
    buckets = defaultdict(list); keep = []; conflict = set()
    for i, h in enumerate(d.phash):
        match = None
        for key in bucket_keys(h, DUP_DIST):
            for j in buckets[key]:
                if (h ^ d.phash[j]).bit_count() <= DUP_DIST: match = j; break
            if match is not None: break
        if match is None:
            keep.append(i)
            for key in bucket_keys(h, DUP_DIST): buckets[key].append(i)
        elif d.label[match] != d.label[i]:
            conflict.add(match)
    log[f"{task}_label_conflict"] = len(conflict)
    d = d.loc[[i for i in keep if i not in conflict]].reset_index(drop=True)

    parent = list(range(len(d)))
    def f(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    buckets = defaultdict(list)
    for i, h in enumerate(d.phash):
        for key in bucket_keys(h, GROUP_DIST):
            for j in buckets[key]:
                if (h ^ d.phash[j]).bit_count() <= GROUP_DIST: parent[f(i)] = f(j)
            buckets[key].append(i)
    d["group"] = [f"{task}_{f(i)}" for i in range(len(d))]
    kept.append(d)
df = pd.concat(kept, ignore_index=True)
log["duplicates_removed"] = before - len(df) - sum(v for k, v in log.items() if k.endswith("conflict"))
print(dict(log), "| after dedupe:", len(df))
print(df.groupby(["task","label"]).size().to_string())

In [ ]:
d0 = pd.DataFrame(recs)
print("Exact copies (same file bytes):", int(d0.duplicated("sha").sum()))
print(d0[d0.duplicated("sha", keep=False)].groupby("label").size().to_string())

# show 6 near-duplicate pairs from loose_smut
ls = d0[d0.label == "loose_smut"].reset_index(drop=True)
pairs, seen = [], {}
for i, h in enumerate(ls.phash):
    for key in bucket_keys(h, DUP_DIST):
        if key in seen and (h ^ ls.phash[seen[key]]).bit_count() <= DUP_DIST and ls.sha[i] != ls.sha[seen[key]]:
            pairs.append((seen[key], i)); break
        seen.setdefault(key, i)
    if len(pairs) == 6: break
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for c, (a, b) in enumerate(pairs):
    ax[0, c].imshow(Image.open(ls.tmp[a])); ax[0, c].set_title(ls.source[a][:18], fontsize=7)
    ax[1, c].imshow(Image.open(ls.tmp[b])); ax[1, c].set_title(ls.source[b][:18], fontsize=7)
for a in ax.flat: a.axis("off")
plt.suptitle("Top vs bottom = pairs marked as duplicates"); plt.show()

## Step 6: Remove blurry images
Removes images below `BLUR_MIN` and shows the 12 least-blurry ones being removed, so we can check the limit.
If good photos appear here, lower `BLUR_MIN` in Step 1 and re-run from Step 4.

In [ ]:
blurry = df[df.blur < BLUR_MIN]
log["blurry"] = len(blurry)
show = blurry.sort_values("blur", ascending=False).head(12)
if len(show):
    fig, ax = plt.subplots(2, 6, figsize=(15, 5))
    for a in ax.flat: a.axis("off")
    for a, r in zip(ax.flat, show.itertuples()):
        a.imshow(Image.open(r.tmp)); a.set_title(f"{r.label}\n{r.blur:.0f}", fontsize=8)
    plt.suptitle("Least-blurry images being REMOVED (check these)"); plt.show()
if len(blurry): print("Blurry by class:\n", blurry.groupby(["task","label"]).size().to_string())
df = df[df.blur >= BLUR_MIN].reset_index(drop=True)
print("removed", len(blurry), "| remaining", len(df))

## Step 7: Drop tiny classes and split into train / val / test
Removes classes with fewer than `MIN_PER_CLASS` images, then splits every class 70 / 15 / 15.
Similar photos (same group from Step 5) always go into the same split, so the test score is honest.

In [ ]:
counts = df.groupby(["task","label"]).size()
small = counts[(counts < MIN_PER_CLASS) & (counts.index.get_level_values(0) != "not_wheat")]
if len(small): print("DROPPED (fewer than", MIN_PER_CLASS, "images):\n", small.to_string())
df = df[~df.set_index(["task","label"]).index.isin(small.index)].reset_index(drop=True)

df["split"] = ""
for _, d in df.groupby(["task","label"]):
    groups = d.groupby("group").size().sample(frac=1, random_state=SEED)
    n, acc, out = len(d), 0, {}
    for g, c in groups.items():
        out[g] = "train" if acc < SPLIT[0]*n else ("val" if acc < (SPLIT[0]+SPLIT[1])*n else "test")
        acc += c
    df.loc[d.index, "split"] = d.group.map(out)
print(pd.crosstab([df.task, df.label], df.split, margins=True).to_string())

## Step 8: Save the clean dataset
Copies every clean image to `/kaggle/working/fasl_clean/<task>/<split>/<label>/` and writes:
- `manifest.csv`: one row per image (source, label, split, hash, blur score, size)
- `data_report.json`: counts before and after each cleaning step

In [ ]:
if OUT.exists(): shutil.rmtree(OUT)
paths = []
for r in df.itertuples():
    p = OUT / r.task / r.split / r.label / f"{r.sha[:16]}.jpg"
    p.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.tmp, p); paths.append(str(p.relative_to(OUT)))
df["path"] = paths
out_df = df.drop(columns=["tmp"]).assign(phash=df.phash.map(lambda h: f"{h:016x}"))
out_df.to_csv(OUT / "manifest.csv", index=False)

report = {
  "raw_images": int(len(raw)),
  "removed": {k: int(v) for k, v in log.items()},
  "final_images": int(len(df)),
  "dropped_small_classes": [list(i) for i in small.index],
  "per_class": {f"{t}/{l}": {s: int(n) for s, n in g.split.value_counts().items()} for (t, l), g in df.groupby(["task","label"])},
  "per_source": {k: int(v) for k, v in df.source.value_counts().items()},
  "settings": dict(MAX_SIDE=MAX_SIDE, MIN_SIDE=MIN_SIDE, BLUR_MIN=BLUR_MIN, DUP_DIST=DUP_DIST, GROUP_DIST=GROUP_DIST,
                   MIN_PER_CLASS=MIN_PER_CLASS, MAX_PER_SOURCE_CLASS=MAX_PER_SOURCE_CLASS, MAX_NOT_WHEAT=MAX_NOT_WHEAT,
                   SPLIT=SPLIT, SEED=SEED),
}
(OUT / "data_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## Step 9: Visual check
Shows 8 random images from every class. Look for wrong labels or junk images and note them down.

In [ ]:
for (t, l), g in df.groupby(["task","label"]):
    s = g.sample(min(8, len(g)), random_state=SEED)
    fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
    for a in ax: a.axis("off")
    for a, r in zip(ax, s.itertuples()): a.imshow(Image.open(OUT / r.path))
    fig.suptitle(f"{t} / {l}  ({len(g)})"); plt.show()

## Step 10: Next
1. Top right: **Save Version → Save & Run All (Commit)**.
2. When it finishes, open the version → **Output** → **New Dataset**, name it `fasl-clean`.
3. Notebook 2 (training) uses `fasl-clean` as its input.